## Notebook to start a spark instance in an ArcGIS Pro Session.

### THE BELOW HAS TO BE THE FIRST IMPORT !

In [1]:
from spark_esri import spark_start, spark_stop

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [3]:
import os

import arcpy
import geofunctions as S

### Check if the jar exists

In [4]:
spark_jars = os.path.expanduser("~/geofunctions-0.15.jar")
os.path.exists(spark_jars)

True

In [5]:
config = {
    "spark.driver.memory": "32G",
    "spark.executor.memory": "32G",
    "spark.jars": spark_jars,
}
spark = spark_start(config=config)

In [6]:
spark.version

'3.5.3-esri-1'

### Register the geo functions with SparkSQL.

In [7]:
S.st_register_functions()

### Sample usage. Create random points in the current map extent and place them on the Map.

In [8]:
project = arcpy.mp.ArcGISProject("CURRENT")

extent = project.activeView.camera.getExtent()
xmin = extent.XMin
ymin = extent.YMin
dx = extent.XMax - extent.XMin
dy = extent.YMax - extent.YMin

In [9]:
(
    spark
    # Create 1000 points.
    .range(1000)
    # Get random x values
    .withColumn("x", F.rand() * dx + xmin)
    # Get random y values.
    .withColumn("y", F.rand() * dy + ymin)
    # Create a point geometry from implicit x/y.
    .withColumn("geom", S.st_point())
    # Convert the spark dataframe to a feature class to be displayed in the TOC.
    .to_feature_class(
        "MyPoints",
        sp_ref = -1,  # Use current map SR.
    )
)

### Create a spark dataframe from a feature class.

In [10]:
desc = arcpy.Describe("MyPoints")
fields = [f.name for f in desc.fields]
fields

['OBJECTID', 'Shape', 'id', 'x', 'y']

In [11]:
df = S.to_spark(desc.name, fields=fields)

In [12]:
df.printSchema()

root
 |-- OBJECTID: long (nullable = true)
 |-- Shape: binary (nullable = true)
 |-- id: long (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)



In [13]:
df.count()

1000

In [14]:
(df
     .select(
        "OBJECTID",
        S.st_astext("Shape"),
    )
     .where(F.col("OBJECTID").between(10,15))
     .show(truncate=False)
)

+--------+---------------------------------------------+
|OBJECTID|text                                         |
+--------+---------------------------------------------+
|10      |POINT (-79.74204462899996 25.626752623000073)|
|11      |POINT (-79.86513551099995 25.80893865400003) |
|12      |POINT (-79.85452532999994 25.742082285000038)|
|13      |POINT (-79.84645846099994 25.675273084000025)|
|14      |POINT (-80.13839083599999 25.843703101000074)|
|15      |POINT (-79.73303490099994 25.681560731000047)|
+--------+---------------------------------------------+



### Create a Spark SQL table from the dataframe and apply some SQL functions.

In [15]:
df.createOrReplaceTempView("v")

In [16]:
spark.sql("""
select OBJECTID,ST_ASTEXT(Shape) as WKT
from v
where OBJECTID between 10 and 15
""").show(truncate=False)

+--------+---------------------------------------------+
|OBJECTID|WKT                                          |
+--------+---------------------------------------------+
|10      |POINT (-79.74204462899996 25.626752623000073)|
|11      |POINT (-79.86513551099995 25.80893865400003) |
|12      |POINT (-79.85452532999994 25.742082285000038)|
|13      |POINT (-79.84645846099994 25.675273084000025)|
|14      |POINT (-80.13839083599999 25.843703101000074)|
|15      |POINT (-79.73303490099994 25.681560731000047)|
+--------+---------------------------------------------+

